In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np

import torchvision.transforms as transforms
from torchvision.utils import save_image

from torch.utils.data import DataLoader
from torchvision import datasets

import torch.nn as nn
import torch

N_EPOCHS = 200
BATCH_SIZE = 64
LR = 0.0002
B1 = 0.5
B2 = 0.999
LATENT_DIM = 100
IMG_SIZE = 28
CHANNELS = 1
SAMPLE_INTERVAL = 400
IMG_DIR = "/content/drive/MyDrive/gan_images_mine"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(IMG_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")

img_shape = (CHANNELS, IMG_SIZE, IMG_SIZE)

class Generator(nn.Module):
  def __init__(self):
    super(Generator, self).__init__()

    def block(in_feat, out_feat, normalize=True):
      layers = [nn.Linear(in_feat, out_feat)]

      if normalize:
        layers.append(nn.BatchNorm1d(out_feat, 0.8))

      layers.append(nn.LeakyReLU(0.2, inplace=True))

      return layers

    self.model=nn.Sequential(
            *block(LATENT_DIM, 128, normalize=False),
            *block(128, 256),
            *block(256, 512),
            *block(512, 1024),
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()
    )

  def forward(self, z):
        img = self.model(z)

        img = img.view(img.size(0), *img_shape)

        return img

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(int(np.prod(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)

        validity = self.model(img_flat)

        return validity

adversarial_loss = nn.BCELoss()

generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)

dataloader = DataLoader(
    datasets.MNIST(
        "./data/mnist",
        train=True,
        download=True,

        transform=transforms.Compose([
            transforms.Resize(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ]),
    ),

    batch_size=BATCH_SIZE,
    shuffle=True,
)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=LR, betas=(B1, B2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(B1, B2))

for epoch in range(N_EPOCHS):

    for i, (imgs, _) in enumerate(dataloader):

        batch_size = imgs.size(0)

        valid = torch.ones(batch_size, 1, device=DEVICE)
        fake = torch.zeros(batch_size, 1, device=DEVICE)

        real_imgs = imgs.to(DEVICE)


        optimizer_G.zero_grad()

        z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
        gen_imgs = generator(z)

        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()


        optimizer_D.zero_grad()

        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        print(
            f"[Epoch {epoch}/{N_EPOCHS}] "
            f"[Batch {i}/{len(dataloader)}] "
            f"[D loss: {d_loss.item():.4f}] "
            f"[G loss: {g_loss.item():.4f}]"
        )


        batches_done = epoch * len(dataloader) + i

        if batches_done % SAMPLE_INTERVAL == 0:

            save_image(
                gen_imgs.data[:25],
                f"{IMG_DIR}/{batches_done}.png",
                nrow=5,
                normalize=True
            )

print("\nTraining complete.")
print(f"Generated images saved in ./{IMG_DIR}/")


Mounted at /content/drive
Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.67MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.21MB/s]


Streaming output truncated to the last 5000 lines.
[Epoch 194/200] [Batch 631/938] [D loss: 0.3285] [G loss: 1.7470]
[Epoch 194/200] [Batch 632/938] [D loss: 0.3041] [G loss: 1.5767]
[Epoch 194/200] [Batch 633/938] [D loss: 0.3146] [G loss: 1.7373]
[Epoch 194/200] [Batch 634/938] [D loss: 0.3113] [G loss: 2.0684]
[Epoch 194/200] [Batch 635/938] [D loss: 0.2909] [G loss: 1.9389]
[Epoch 194/200] [Batch 636/938] [D loss: 0.2887] [G loss: 1.5532]
[Epoch 194/200] [Batch 637/938] [D loss: 0.2779] [G loss: 1.9166]
[Epoch 194/200] [Batch 638/938] [D loss: 0.3232] [G loss: 1.9957]
[Epoch 194/200] [Batch 639/938] [D loss: 0.2603] [G loss: 1.6138]
[Epoch 194/200] [Batch 640/938] [D loss: 0.2976] [G loss: 2.0873]
[Epoch 194/200] [Batch 641/938] [D loss: 0.3474] [G loss: 2.1589]
[Epoch 194/200] [Batch 642/938] [D loss: 0.3959] [G loss: 1.7514]
[Epoch 194/200] [Batch 643/938] [D loss: 0.3631] [G loss: 1.6320]
[Epoch 194/200] [Batch 644/938] [D loss: 0.2852] [G loss: 1.9840]
[Epoch 194/200] [Batch 64